# DEAP Preprocessing: Resampling, Alignment, Missing Modalities, Sliding Window

This notebook covers the preprocessing steps for multimodal DEAP data, including timestamp alignment, handling missing modalities, and sliding window segmentation for continuous prediction.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = '../src/data'

physio_path = os.path.join(DATA_DIR, 'physio_features.csv')
audio_path = os.path.join(DATA_DIR, 'audio_features.csv')

physio_features_df = pd.read_csv(physio_path) if os.path.exists(physio_path) else None
audio_features_df = pd.read_csv(audio_path) if os.path.exists(audio_path) else None

print('Physio features shape:', physio_features_df.shape if physio_features_df is not None else 'Not loaded')
print('Audio features shape:', audio_features_df.shape if audio_features_df is not None else 'Not loaded')

Physio features shape: (1280, 32)
Audio features shape: (40, 4)


In [2]:
# Resample and align timestamps
# Robust alignment of features using index or trial column
if physio_features_df is not None and audio_features_df is not None:
    # If both have a 'trial' column, merge on it; else, use index
    if 'trial' in physio_features_df.columns and 'trial' in audio_features_df.columns:
        aligned_df = pd.merge(physio_features_df, audio_features_df, on='trial', suffixes=('_physio', '_audio'))
    else:
        min_len = min(len(physio_features_df), len(audio_features_df))
        aligned_df = pd.concat([
            physio_features_df.iloc[:min_len].reset_index(drop=True),
            audio_features_df.iloc[:min_len].reset_index(drop=True)
        ], axis=1)
    print('Aligned features shape:', aligned_df.shape)
else:
    print('Alignment skipped: features not loaded.')

Aligned features shape: (40, 36)


In [3]:
# Impute missing values: mean for numeric, mode for non-numeric columns

def impute_df(df):
    if df is not None:
        # Mean for numeric columns
        for col in df.select_dtypes(include=[np.number]).columns:
            mean_val = df[col].mean(skipna=True)
            df[col] = df[col].fillna(mean_val)
        # Mode for non-numeric columns
        for col in df.select_dtypes(exclude=[np.number]).columns:
            mode_val = df[col].mode(dropna=True)
            if not mode_val.empty:
                df[col] = df[col].fillna(mode_val[0])
    return df

physio_features_df = impute_df(physio_features_df)
audio_features_df = impute_df(audio_features_df)

print('Missing values handled (mean for numeric, mode for non-numeric columns).')

Missing values handled (mean for numeric, mode for non-numeric columns).


In [4]:
# Sliding window segmentation
WINDOW_SIZE = 10  # Sample window size
STEP_SIZE = 5     # Sample step size

def sliding_window(df, window_size, step_size):
    windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        windows.append(df.iloc[start:start+window_size].values)
    return np.array(windows)

if physio_features_df is not None:
    physio_windows = sliding_window(physio_features_df, WINDOW_SIZE, STEP_SIZE)
    print('Physio windows shape:', physio_windows.shape)
if audio_features_df is not None:
    audio_windows = sliding_window(audio_features_df, WINDOW_SIZE, STEP_SIZE)
    print('Audio windows shape:', audio_windows.shape)

Physio windows shape: (255, 10, 32)
Audio windows shape: (7, 10, 4)


In [5]:
physio_features_df.to_csv('../db/physio_features.csv', index=False)
audio_features_df.to_csv('../db/audio_features.csv', index=False)
print('Feature CSVs saved: ../db/physio_features.csv, ../db/audio_features.csv')

Feature CSVs saved: ../db/physio_features.csv, ../db/audio_features.csv
